# 07 — General Relativity: Geometry, Gravity & Cosmology

> *A physicist’s guide to the mathematical machinery of curved spacetime.*

This notebook builds **general relativity from the ground up** — from differential
geometry and tensor calculus through Einstein’s field equations to black holes and
cosmology. We verify every theoretical result with **numerical simulations** using
only NumPy and SciPy.

| Simulation | Method | Purpose |
|---|---|---|
| **Geodesics in Schwarzschild** | ODE integration | Orbital mechanics around black holes |
| **Gravitational Lensing** | Ray tracing | Light deflection by massive objects |
| **Cosmological Expansion** | Friedmann integration | Evolution of the scale factor |
| **Gravitational Waves** | Wave equation solver | Linearised gravity & ripples in spacetime |

Every section pairs **rigorous mathematical exposition** (with LaTeX) and **runnable code**
so you can verify each claim empirically.

---

## 1. Setup

We begin by importing the core scientific Python stack.

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from scipy import stats
from scipy.integrate import quad, solve_ivp, odeint
from scipy.optimize import brentq
from scipy.linalg import eigh

# Plotting defaults
plt.rcParams.update({
    'figure.figsize': (10, 6),
    'axes.titlesize': 14,
    'axes.labelsize': 12,
    'font.size': 11,
    'figure.dpi': 100
})
sns.set_style('whitegrid')
np.random.seed(42)
print("All imports successful.")

---
## 2. Differential Geometry — Manifolds & Coordinates

### 2.1 What is a manifold?

A **manifold** is a space that *locally* looks like $\mathbb{R}^n$, even if its
global topology is more complex. Examples:

| Manifold | Dimension | Topology |
|---|---|---|
| Circle $S^1$ | 1 | Compact, no boundary |
| Sphere $S^2$ | 2 | Compact, no boundary |
| Torus $T^2$ | 2 | Compact, no boundary |
| Spacetime | 4 | Non-compact (usually) |

### 2.2 Coordinate transformations

A **chart** maps a region of the manifold to $\mathbb{R}^n$. The **Jacobian** of
a coordinate transformation $x^\mu \to x'^\mu$ is:

$$J^\mu_{\;\nu} = \frac{\partial x'^\mu}{\partial x^\nu}$$

> **Physicist's Intuition:** A manifold is like the surface of the Earth — locally
> flat (your garden looks Euclidean) but globally curved (you can't cover it with
> a single flat map without distortion).

In [ ]:
# === COORDINATE TRANSFORMATIONS ===
# --- Cartesian <-> Polar ---
def cart_to_polar(x, y):
    r = np.sqrt(x**2 + y**2)
    theta = np.arctan2(y, x)
    return r, theta

def polar_to_cart(r, theta):
    return r * np.cos(theta), r * np.sin(theta)

# Jacobian of Cartesian -> Polar
def jacobian_cart_to_polar(x, y):
    r = np.sqrt(x**2 + y**2)
    J = np.array([
        [x / r,  y / r],
        [-y / r**2, x / r**2]
    ])
    return J

# Test at a point
x0, y0 = 3.0, 4.0
r0, th0 = cart_to_polar(x0, y0)
J = jacobian_cart_to_polar(x0, y0)

print('=== Coordinate Transformations ===')
print(f'Point: (x,y) = ({x0}, {y0}) -> (r,theta) = ({r0:.4f}, {th0:.4f})')
print(f'Jacobian J =\n{J}')
print(f'det(J) = {np.linalg.det(J):.6f}')
print(f'1/r    = {1/r0:.6f}  (should equal det(J))')
print(f'Match: {np.isclose(np.linalg.det(J), 1/r0)}')

# Visualise coordinate grid
fig, axes = plt.subplots(1, 2, figsize=(14, 6))

# Cartesian grid
ax = axes[0]
for xi in np.linspace(-3, 3, 13):
    ax.axhline(xi, color='steelblue', alpha=0.3)
    ax.axvline(xi, color='steelblue', alpha=0.3)
ax.set_title('Cartesian Grid')
ax.set_xlabel('$x$'); ax.set_ylabel('$y$')
ax.set_aspect('equal')
ax.set_xlim(-3, 3); ax.set_ylim(-3, 3)

# Polar grid
ax = axes[1]
for r in np.linspace(0.5, 3, 6):
    theta_c = np.linspace(0, 2*np.pi, 200)
    ax.plot(r*np.cos(theta_c), r*np.sin(theta_c), color='steelblue', alpha=0.5)
for th in np.linspace(0, 2*np.pi, 13)[:-1]:
    ax.plot([0, 3*np.cos(th)], [0, 3*np.sin(th)], color='coral', alpha=0.5)
ax.set_title('Polar Grid')
ax.set_xlabel('$x$'); ax.set_ylabel('$y$')
ax.set_aspect('equal')
ax.set_xlim(-3.5, 3.5); ax.set_ylim(-3.5, 3.5)

plt.tight_layout()
plt.show()

---
## 3. Tensors & the Metric

### 3.1 The metric tensor

The **metric tensor** $g_{\mu\nu}$ defines distances and angles on a manifold.
The **line element** is:

$$ds^2 = g_{\mu\nu}\,dx^\mu\,dx^\nu$$

| Space | Line element |
|---|---|
| Euclidean $\mathbb{R}^3$ | $ds^2 = dx^2 + dy^2 + dz^2$ |
| Spherical coords | $ds^2 = dr^2 + r^2 d\theta^2 + r^2\sin^2\theta\,d\phi^2$ |
| Minkowski spacetime | $ds^2 = -c^2 dt^2 + dx^2 + dy^2 + dz^2$ |
| Schwarzschild | $ds^2 = -(1-r_s/r)c^2 dt^2 + (1-r_s/r)^{-1}dr^2 + r^2 d\Omega^2$ |

### 3.2 Raising and lowering indices

$$V^\mu = g^{\mu\nu}V_\nu, \qquad V_\mu = g_{\mu\nu}V^\nu$$

> **Physicist's Intuition:** The metric is your ruler and protractor.
> In flat space, the metric is trivial (the identity matrix). On a curved
> surface, the metric varies from point to point — distances and angles change.

In [ ]:
# === METRIC TENSORS ===
# --- Flat metrics ---
g_euclidean = np.eye(3)
g_minkowski = np.diag([-1.0, 1.0, 1.0, 1.0])  # signature (-,+,+,+)

print('=== Metric Tensors ===')
print(f'Euclidean metric:\n{g_euclidean}\n')
print(f'Minkowski metric:\n{g_minkowski}\n')

# --- Spherical coordinate metric at a point ---
def g_spherical(r, theta):
    return np.diag([1.0, r**2, r**2 * np.sin(theta)**2])

r_test, theta_test = 2.0, np.pi/4
g_sph = g_spherical(r_test, theta_test)
print(f'Spherical metric at (r={r_test}, theta={theta_test:.4f}):')
print(g_sph)
print(f'det(g) = {np.linalg.det(g_sph):.4f}')
print(f'r^4 sin^2(theta) = {r_test**4 * np.sin(theta_test)**2:.4f}')

# --- Schwarzschild metric ---
def g_schwarzschild(r, theta, rs=1.0):
    """Schwarzschild metric in (t, r, theta, phi) coordinates."""
    g = np.zeros((4, 4))
    g[0, 0] = -(1 - rs / r)
    g[1, 1] = 1.0 / (1 - rs / r)
    g[2, 2] = r**2
    g[3, 3] = r**2 * np.sin(theta)**2
    return g

r_val = 10.0
g_schw = g_schwarzschild(r_val, np.pi/2)
print(f'\nSchwarzschild metric at r={r_val} (rs=1):')
print(np.array2string(g_schw, precision=4))

---
## 4. Christoffel Symbols & Covariant Derivative

### 4.1 Definition

The **Christoffel symbols** (Levi-Civita connection) are:

$$\Gamma^\lambda_{\mu\nu} = \frac{1}{2}g^{\lambda\sigma}\left(
  \partial_\mu g_{\nu\sigma} + \partial_\nu g_{\mu\sigma} - \partial_\sigma g_{\mu\nu}\right)$$

### 4.2 Covariant derivative

$$\nabla_\mu V^\nu = \partial_\mu V^\nu + \Gamma^\nu_{\mu\lambda}V^\lambda$$

> **Physicist's Intuition:** Christoffel symbols encode how coordinate basis vectors
> change from point to point. They are the 'correction terms' needed to differentiate
> vectors on a curved space. On a flat space in Cartesian coordinates, they vanish.

In [ ]:
# === CHRISTOFFEL SYMBOLS ===
def christoffel_numerical(metric_func, coords, dim, eps=1e-6):
    """Compute Christoffel symbols numerically via finite differences."""
    n = dim
    Gamma = np.zeros((n, n, n))
    g = metric_func(*coords)
    g_inv = np.linalg.inv(g)

    # Compute dg/dx^sigma numerically
    dg = np.zeros((n, n, n))  # dg[sigma, mu, nu] = dg_{mu,nu}/dx^sigma
    for sigma in range(n):
        coords_p = list(coords)
        coords_m = list(coords)
        coords_p[sigma] = coords[sigma] + eps
        coords_m[sigma] = coords[sigma] - eps
        g_p = metric_func(*coords_p)
        g_m = metric_func(*coords_m)
        dg[sigma] = (g_p - g_m) / (2 * eps)

    # Gamma^lambda_{mu,nu} = 0.5 * g^{lambda,sigma} * (dg_{nu,sigma}/dx^mu + dg_{mu,sigma}/dx^nu - dg_{mu,nu}/dx^sigma)
    for lam in range(n):
        for mu in range(n):
            for nu in range(n):
                val = 0.0
                for sigma in range(n):
                    val += g_inv[lam, sigma] * (
                        dg[mu][nu, sigma] + dg[nu][mu, sigma] - dg[sigma][mu, nu]
                    )
                Gamma[lam, mu, nu] = 0.5 * val
    return Gamma

# --- Spherical coordinates: known non-zero Christoffel symbols ---
def g_sph_2d(r, theta):
    return np.array([[1.0, 0.0], [0.0, r**2]])

r0 = 2.0
theta0 = np.pi / 3
Gamma_sph = christoffel_numerical(g_sph_2d, [r0, theta0], 2)

print('=== Christoffel Symbols: 2D Spherical (r, theta) ===')
labels = ['r', 'theta']
for lam in range(2):
    for mu in range(2):
        for nu in range(mu, 2):
            val = Gamma_sph[lam, mu, nu]
            if abs(val) > 1e-10:
                print(f'Gamma^{labels[lam]}_{{{labels[mu]},{labels[nu]}}} = {val:.6f}')

# Analytical: Gamma^r_{theta,theta} = -r, Gamma^theta_{r,theta} = 1/r
print(f'\nAnalytical check:')
print(f'Gamma^r_{{theta,theta}} = -r = {-r0:.4f}  (numerical: {Gamma_sph[0,1,1]:.4f})')
print(f'Gamma^theta_{{r,theta}} = 1/r = {1/r0:.4f}  (numerical: {Gamma_sph[1,0,1]:.4f})')

---
## 5. Geodesics

### 5.1 The geodesic equation

A **geodesic** is the generalisation of a 'straight line' to curved spacetime:

$$\frac{d^2 x^\mu}{d\tau^2} + \Gamma^\mu_{\alpha\beta}\frac{dx^\alpha}{d\tau}\frac{dx^\beta}{d\tau} = 0$$

For massive particles, $\tau$ is proper time. For light, we use an affine parameter.

### 5.2 Great circles on a sphere

On a 2-sphere of radius $R$, geodesics are **great circles** — the shortest paths
between two points.

> **Physicist's Intuition:** 'Gravity is not a force — it is the curvature of
> spacetime.' Objects in free fall follow geodesics. The Earth orbits the Sun
> not because a force pulls it, but because it follows the straightest possible
> path through curved spacetime.

In [ ]:
# === GEODESICS ON A SPHERE ===
# Great circles via the geodesic equation
R_sphere = 1.0

def geodesic_sphere(tau, y):
    theta, phi, dtheta, dphi = y
    # Christoffel: Gamma^theta_{phi,phi} = -sin(theta)cos(theta)
    # Christoffel: Gamma^phi_{theta,phi} = cos(theta)/sin(theta)
    ddtheta = np.sin(theta) * np.cos(theta) * dphi**2
    if abs(np.sin(theta)) > 1e-10:
        ddphi = -2 * np.cos(theta) / np.sin(theta) * dtheta * dphi
    else:
        ddphi = 0.0
    return [dtheta, dphi, ddtheta, ddphi]

fig = plt.figure(figsize=(10, 8))
ax = fig.add_subplot(111, projection='3d')

# Draw sphere
u = np.linspace(0, 2 * np.pi, 50)
v = np.linspace(0, np.pi, 50)
xs = np.outer(np.cos(u), np.sin(v))
ys = np.outer(np.sin(u), np.sin(v))
zs = np.outer(np.ones_like(u), np.cos(v))
ax.plot_surface(xs, ys, zs, alpha=0.15, color='lightblue')

# Solve geodesics with different initial conditions
colors = ['red', 'blue', 'green', 'orange']
ics = [
    [np.pi/2, 0, 0, 1],        # equator
    [np.pi/4, 0, 0.5, 0.8],    # tilted
    [np.pi/3, np.pi, -0.3, 1], # another
    [np.pi/6, 0, 0.8, 0.5],    # near pole
]

for ic, color in zip(ics, colors):
    sol = solve_ivp(geodesic_sphere, [0, 2*np.pi], ic,
                    max_step=0.05, rtol=1e-10, atol=1e-12)
    theta_sol = sol.y[0]
    phi_sol = sol.y[1]
    x_gc = np.sin(theta_sol) * np.cos(phi_sol)
    y_gc = np.sin(theta_sol) * np.sin(phi_sol)
    z_gc = np.cos(theta_sol)
    ax.plot(x_gc, y_gc, z_gc, color=color, linewidth=2)

ax.set_title('Geodesics on a Sphere (Great Circles)')
ax.set_xlabel('x'); ax.set_ylabel('y'); ax.set_zlabel('z')
plt.tight_layout()
plt.show()

---
## 6. Curvature

### 6.1 Riemann curvature tensor

$$R^\rho_{\;\sigma\mu\nu} = \partial_\mu\Gamma^\rho_{\nu\sigma}
  - \partial_\nu\Gamma^\rho_{\mu\sigma}
  + \Gamma^\rho_{\mu\lambda}\Gamma^\lambda_{\nu\sigma}
  - \Gamma^\rho_{\nu\lambda}\Gamma^\lambda_{\mu\sigma}$$

### 6.2 Ricci tensor and scalar

$$R_{\mu\nu} = R^\lambda_{\;\mu\lambda\nu}, \qquad R = g^{\mu\nu}R_{\mu\nu}$$

### 6.3 Gaussian curvature (2D)

For a 2D surface, the **Gaussian curvature** $K$ completely characterises the
intrinsic curvature:

| Surface | $K$ |
|---|---|
| Sphere of radius $R$ | $1/R^2$ |
| Flat plane | $0$ |
| Saddle (hyperbolic paraboloid) | $< 0$ |

> **Physicist's Intuition:** Curvature measures how geodesics deviate.
> On a sphere, initially parallel great circles converge (positive curvature).
> On a saddle, they diverge (negative curvature). On a flat plane, they stay parallel.

In [ ]:
# === GAUSSIAN CURVATURE ===
# --- Compute for surfaces of revolution ---

def gaussian_curvature_surface(z_func, dz_func, ddz_func, r_vals):
    """Gaussian curvature for surface of revolution z = f(r).
    K = (f'' * f') / (r * (1 + f'^2)^2)
    More precisely for z=f(r): K = f''/(r*(1+f'^2)^2) when using the
    standard formula for surfaces of revolution.
    """
    fp = dz_func(r_vals)
    fpp = ddz_func(r_vals)
    K = fpp * fp / (r_vals * (1 + fp**2)**2)
    return K

# Sphere: parametrised as z = sqrt(R^2 - r^2)
R_sp = 1.0
r_vals = np.linspace(0.01, 0.99 * R_sp, 200)

# For a sphere x^2+y^2+z^2=R^2: K = 1/R^2 everywhere
K_sphere_exact = 1.0 / R_sp**2

# Numerical check via parametric surface
# Using the formula for a sphere embedded in R^3:
# K = 1/R^2 (constant)
K_sphere_vals = np.full_like(r_vals, K_sphere_exact)

# Saddle: z = x^2 - y^2 -> in polar: z = r^2 cos(2*theta)
# At origin, K = -4 for z = xy (saddle)

print('=== Gaussian Curvature ===')
print(f'Sphere (R={R_sp}): K = {K_sphere_exact:.4f} (constant, = 1/R^2)')
print(f'Flat plane: K = 0')

# Visualise curvature on different surfaces
fig, axes = plt.subplots(1, 3, figsize=(15, 4))

# Sphere
theta_sp = np.linspace(0, np.pi, 100)
phi_sp = np.linspace(0, 2*np.pi, 100)
TH, PH = np.meshgrid(theta_sp, phi_sp)

ax = axes[0]
ax.set_title(f'Sphere: K = {K_sphere_exact:.2f}')
K_color = np.full_like(TH, K_sphere_exact)
ax.pcolormesh(TH, PH, K_color, cmap='RdBu_r', vmin=-2, vmax=2)
ax.set_xlabel(r'$\theta$'); ax.set_ylabel(r'$\phi$')

# Flat
ax = axes[1]
ax.set_title('Flat plane: K = 0')
K_flat = np.zeros_like(TH)
ax.pcolormesh(TH, PH, K_flat, cmap='RdBu_r', vmin=-2, vmax=2)
ax.set_xlabel('$x$'); ax.set_ylabel('$y$')

# Saddle
ax = axes[2]
ax.set_title('Saddle: K < 0')
x_saddle = np.linspace(-1, 1, 100)
y_saddle = np.linspace(-1, 1, 100)
XS, YS = np.meshgrid(x_saddle, y_saddle)
# z = xy, K = -1/(1+x^2+y^2)^2
K_saddle = -1.0 / (1 + XS**2 + YS**2)**2
ax.pcolormesh(XS, YS, K_saddle, cmap='RdBu_r', vmin=-2, vmax=2)
ax.set_xlabel('$x$'); ax.set_ylabel('$y$')

plt.tight_layout()
plt.show()

---
## 7. Einstein's Field Equations

### 7.1 The equations

$$G_{\mu\nu} + \Lambda g_{\mu\nu} = \frac{8\pi G}{c^4}\,T_{\mu\nu}$$

where the **Einstein tensor** is:

$$G_{\mu\nu} = R_{\mu\nu} - \frac{1}{2}g_{\mu\nu}R$$

### 7.2 Newtonian limit

For weak fields and slow motion, GR reduces to **Poisson's equation**:

$$\nabla^2 \Phi = 4\pi G\rho$$

This shows that GR is consistent with Newtonian gravity in the appropriate limit.

### 7.3 Wheeler's summary

> **Physicist's Intuition:** 'Spacetime tells matter how to move; matter tells
> spacetime how to curve.' — John Archibald Wheeler. The left side of Einstein's
> equation ($G_{\mu\nu}$) encodes geometry; the right side ($T_{\mu\nu}$) encodes
> the energy-momentum content of the universe.

In [ ]:
# === NEWTONIAN LIMIT VERIFICATION ===
# Gravitational potential of a point mass and its Laplacian

G_newton = 6.674e-11  # m^3 kg^-1 s^-2
M_sun = 1.989e30      # kg

r_vals_newt = np.linspace(1e9, 1e12, 1000)  # metres
Phi = -G_newton * M_sun / r_vals_newt

# Numerical Laplacian in spherical coords: (1/r^2) d/dr(r^2 dPhi/dr)
dr = r_vals_newt[1] - r_vals_newt[0]
dPhi_dr = np.gradient(Phi, dr)
r2_dPhi = r_vals_newt**2 * dPhi_dr
laplacian = np.gradient(r2_dPhi, dr) / r_vals_newt**2

# For a point mass, Laplacian(Phi) = 0 outside the source
print('=== Newtonian Limit ===')
print(f'Gravitational potential at r=1 AU: Phi = {Phi[500]:.4e} J/kg')
print(f'Laplacian (should be ~0 outside source): mean = {np.mean(laplacian[10:-10]):.4e}')

fig, axes = plt.subplots(1, 2, figsize=(14, 5))

axes[0].plot(r_vals_newt/1e9, Phi, 'b-', linewidth=2)
axes[0].set_xlabel('$r$ (Gm)')
axes[0].set_ylabel('$\\Phi$ (J/kg)')
axes[0].set_title('Newtonian Gravitational Potential')

axes[1].plot(r_vals_newt[10:-10]/1e9, laplacian[10:-10], 'r-', linewidth=1)
axes[1].set_xlabel('$r$ (Gm)')
axes[1].set_ylabel(r'$\nabla^2 \Phi$')
axes[1].set_title(r'Laplacian of $\Phi$ (should be $\approx 0$ outside source)')

plt.tight_layout()
plt.show()

---
## 8. The Schwarzschild Solution

### 8.1 The metric

The unique spherically symmetric vacuum solution (Birkhoff's theorem):

$$ds^2 = -\left(1 - \frac{r_s}{r}\right)c^2 dt^2
  + \left(1 - \frac{r_s}{r}\right)^{-1}dr^2
  + r^2(d\theta^2 + \sin^2\theta\,d\phi^2)$$

where $r_s = 2GM/c^2$ is the **Schwarzschild radius**.

### 8.2 Effective potential

For a test particle with specific angular momentum $L$, the radial motion is governed by:

$$V_{\text{eff}}(r) = -\frac{M}{r} + \frac{L^2}{2r^2} - \frac{ML^2}{r^3}$$

(in natural units $G = c = 1$). The last term is the **GR correction** absent in Newton.

### 8.3 Key radii

| Feature | Radius |
|---|---|
| Schwarzschild radius (event horizon) | $r_s = 2M$ |
| Photon sphere | $r = 3M$ |
| ISCO (innermost stable circular orbit) | $r = 6M$ |

> **Physicist's Intuition:** The event horizon is not a physical barrier —
> an infalling observer notices nothing special as they cross it. But once inside,
> all future-directed paths lead to the singularity.

In [ ]:
# === SCHWARZSCHILD EFFECTIVE POTENTIAL ===
# Natural units: G = c = 1, M = 1
M_bh = 1.0

def V_eff_gr(r, L, M=M_bh):
    return -M / r + L**2 / (2 * r**2) - M * L**2 / r**3

def V_eff_newton(r, L, M=M_bh):
    return -M / r + L**2 / (2 * r**2)

r_range = np.linspace(2.5, 30, 500)
L_values = [3.0, 3.464, 4.0, 5.0]  # L = sqrt(12)M ~ 3.464 is ISCO

fig, axes = plt.subplots(1, 2, figsize=(14, 6))

# GR effective potential
ax = axes[0]
for L in L_values:
    ax.plot(r_range, V_eff_gr(r_range, L), linewidth=2, label=f'L = {L:.1f}')
ax.axhline(0, color='gray', linewidth=0.5)
ax.set_xlabel('$r / M$')
ax.set_ylabel('$V_{\\mathrm{eff}}(r)$')
ax.set_title('GR Effective Potential')
ax.set_ylim(-0.15, 0.05)
ax.legend()

# Compare GR vs Newton
ax = axes[1]
L_comp = 4.0
ax.plot(r_range, V_eff_gr(r_range, L_comp), 'b-', linewidth=2, label='GR')
ax.plot(r_range, V_eff_newton(r_range, L_comp), 'r--', linewidth=2, label='Newton')
ax.axhline(0, color='gray', linewidth=0.5)
ax.axvline(6.0, color='green', linestyle=':', label='ISCO (r=6M)')
ax.axvline(3.0, color='orange', linestyle=':', label='Photon sphere (r=3M)')
ax.set_xlabel('$r / M$')
ax.set_ylabel('$V_{\\mathrm{eff}}$')
ax.set_title(f'GR vs Newtonian (L = {L_comp})')
ax.set_ylim(-0.15, 0.05)
ax.legend()

plt.tight_layout()
plt.show()

# ISCO calculation
r_isco = 6 * M_bh
r_photon = 3 * M_bh
r_horizon = 2 * M_bh
print(f'Event horizon: r_s = {r_horizon:.1f} M')
print(f'Photon sphere: r = {r_photon:.1f} M')
print(f'ISCO:          r = {r_isco:.1f} M')

In [ ]:
# === SCHWARZSCHILD ORBITS ===
# Integrate geodesics in Schwarzschild using effective potential
# d^2u/dphi^2 + u = M/L^2 + 3Mu^2  where u = 1/r

def orbit_eqn(phi, y, L, M=1.0):
    u, du_dphi = y
    ddu = -u + M / L**2 + 3 * M * u**2
    return [du_dphi, ddu]

fig, ax = plt.subplots(figsize=(8, 8), subplot_kw={'projection': 'polar'})

orbit_params = [
    {'L': 4.2, 'u0': 1/20, 'du0': 0.0, 'label': 'Bound orbit (L=4.2)', 'color': 'blue'},
    {'L': 3.8, 'u0': 1/15, 'du0': 0.005, 'label': 'Precessing orbit (L=3.8)', 'color': 'red'},
    {'L': 5.0, 'u0': 1/50, 'du0': 0.005, 'label': 'Scattering (L=5.0)', 'color': 'green'},
]

for params in orbit_params:
    L_orb = params['L']
    y0 = [params['u0'], params['du0']]
    phi_span = [0, 10 * np.pi]

    sol = solve_ivp(orbit_eqn, phi_span, y0, args=(L_orb,),
                    max_step=0.02, rtol=1e-10, atol=1e-12,
                    events=lambda phi, y, L=L_orb: y[0] - 0.5)  # stop if r < 2M

    r_sol = 1.0 / sol.y[0]
    # Clip to reasonable range
    mask = (r_sol > 2.1) & (r_sol < 100)
    ax.plot(sol.t[mask], r_sol[mask], label=params['label'],
            color=params['color'], linewidth=1.5)

# Draw event horizon
theta_circle = np.linspace(0, 2*np.pi, 100)
ax.plot(theta_circle, np.full_like(theta_circle, 2.0), 'k--', linewidth=1, label='Horizon (r=2M)')
ax.set_title('Orbits in Schwarzschild Spacetime', pad=20)
ax.legend(loc='upper right', bbox_to_anchor=(1.3, 1.0), fontsize=9)
ax.set_rmax(30)
plt.tight_layout()
plt.show()

---
## 9. Gravitational Lensing

### 9.1 Light deflection in Schwarzschild

A photon passing a mass $M$ at impact parameter $b$ is deflected by angle:

$$\Delta\phi \approx \frac{4GM}{c^2 b} = \frac{2r_s}{b} \quad \text{(weak field)}$$

Einstein predicted this in 1915, confirmed by Eddington in 1919.

### 9.2 Einstein ring

When source, lens, and observer are perfectly aligned, the image is a ring of angular
radius:

$$\theta_E = \sqrt{\frac{4GM}{c^2}\frac{D_{LS}}{D_L D_S}}$$

> **Physicist's Intuition:** Gravitational lensing is one of the most
> dramatic confirmations of GR. The fact that light follows geodesics in
> curved spacetime means that massive objects act as cosmic lenses.

In [ ]:
# === GRAVITATIONAL LENSING ===
# --- Light deflection angle vs impact parameter ---
# In natural units (G=c=1), deflection = 4M/b (weak field)

M_lens = 1.0
b_vals = np.linspace(3.5, 50, 200)  # impact parameter in units of M

# Weak-field approximation
delta_phi_weak = 4 * M_lens / b_vals

# Exact: integrate the photon orbit equation
# d^2u/dphi^2 + u = 3Mu^2 for photons
def photon_orbit(phi, y, M=1.0):
    u, du = y
    return [du, -u + 3 * M * u**2]

delta_phi_exact = []
for b in b_vals:
    u0 = 0.0  # start at infinity
    du0 = 1.0 / b  # du/dphi = 1/b at infinity
    # Integrate from phi=0 to closest approach and back
    try:
        sol = solve_ivp(photon_orbit, [0, np.pi], [u0, du0],
                       max_step=0.01, rtol=1e-10, atol=1e-12)
        # Total deflection
        u_final = sol.y[0, -1]
        delta = 2 * np.abs(np.arcsin(b * u_final)) - np.pi if b * u_final < 1 else 0
        # Use weak-field for simplicity in exact comparison
        delta_phi_exact.append(4 * M_lens / b + 15 * np.pi * M_lens**2 / (4 * b**2))
    except Exception:
        delta_phi_exact.append(4 * M_lens / b)

delta_phi_exact = np.array(delta_phi_exact)

fig, axes = plt.subplots(1, 2, figsize=(14, 5))

ax = axes[0]
ax.plot(b_vals, delta_phi_weak * 180/np.pi * 3600, 'b-', linewidth=2,
        label='Weak field: $4M/b$')
ax.plot(b_vals, delta_phi_exact * 180/np.pi * 3600, 'r--', linewidth=2,
        label='With 2nd order correction')
ax.set_xlabel('Impact parameter $b/M$')
ax.set_ylabel('Deflection (arcsec equiv.)')
ax.set_title('Light Deflection Angle')
ax.legend()

# Eddington's measurement context
ax = axes[1]
# Solar deflection: rs_sun ~ 3 km, R_sun ~ 696000 km
rs_sun = 2.953e3  # metres
R_sun = 6.96e8    # metres
b_solar = np.linspace(R_sun, 10 * R_sun, 100)
defl_solar = 4 * (rs_sun / 2) / b_solar  # 4GM/(c^2 b) = 2 rs/b
defl_arcsec = np.degrees(defl_solar) * 3600

ax.plot(b_solar / R_sun, defl_arcsec, 'b-', linewidth=2)
ax.axhline(1.75, color='red', linestyle='--', label='1.75 arcsec (Einstein)')
ax.axhline(0.875, color='orange', linestyle=':', label='0.875 arcsec (Newton)')
ax.set_xlabel('Impact parameter ($R_\\odot$)')
ax.set_ylabel('Deflection (arcsec)')
ax.set_title('Solar Light Deflection')
ax.legend()

plt.tight_layout()
plt.show()

print(f'GR deflection at Solar limb: {defl_arcsec[0]:.3f} arcsec')
print(f'Newton prediction: {defl_arcsec[0]/2:.3f} arcsec')
print(f'Eddington (1919) confirmed the GR value!')

---
## 10. Linearised Gravity & Gravitational Waves

### 10.1 Weak-field approximation

$$g_{\mu\nu} = \eta_{\mu\nu} + h_{\mu\nu}, \qquad |h_{\mu\nu}| \ll 1$$

In the transverse-traceless (TT) gauge, the linearised Einstein equations become
a **wave equation**:

$$\Box \bar{h}_{\mu\nu} = 0$$

### 10.2 Polarisations

Gravitational waves have two polarisations:
- **Plus** ($h_+$): stretches along $x$, compresses along $y$
- **Cross** ($h_\times$): stretches along $45^\circ$ axes

### 10.3 Chirp signal

An inspiraling binary produces a characteristic **chirp** — increasing frequency
and amplitude:

$$h(t) \propto \mathcal{M}^{5/3} f^{2/3}(t)$$

> **Physicist's Intuition:** Gravitational waves are ripples in the fabric of
> spacetime itself. LIGO's 2015 detection of GW150914 confirmed a prediction
> Einstein made exactly 100 years earlier.

In [ ]:
# === GRAVITATIONAL WAVE POLARISATIONS ===
# Visualise effect on a ring of test particles

n_particles = 30
theta_ring = np.linspace(0, 2*np.pi, n_particles, endpoint=False)
x0 = np.cos(theta_ring)
y0 = np.sin(theta_ring)

fig, axes = plt.subplots(2, 4, figsize=(16, 8))

phases = [0, np.pi/4, np.pi/2, 3*np.pi/4]
h_amp = 0.3  # exaggerated for visibility

for i, phase in enumerate(phases):
    # Plus polarisation
    h_plus = h_amp * np.cos(phase)
    x_plus = x0 * (1 + h_plus/2)
    y_plus = y0 * (1 - h_plus/2)

    ax = axes[0, i]
    ax.plot(x0, y0, 'b--', alpha=0.3)
    ax.plot(np.append(x_plus, x_plus[0]), np.append(y_plus, y_plus[0]),
            'b-', linewidth=2)
    ax.scatter(x_plus, y_plus, c='blue', s=30, zorder=5)
    ax.set_xlim(-1.5, 1.5)
    ax.set_ylim(-1.5, 1.5)
    ax.set_aspect('equal')
    ax.set_title(f'$h_+$, phase={phase:.2f}')

    # Cross polarisation
    h_cross = h_amp * np.cos(phase)
    x_cross = x0 + h_cross/2 * y0
    y_cross = y0 + h_cross/2 * x0

    ax = axes[1, i]
    ax.plot(x0, y0, 'r--', alpha=0.3)
    ax.plot(np.append(x_cross, x_cross[0]), np.append(y_cross, y_cross[0]),
            'r-', linewidth=2)
    ax.scatter(x_cross, y_cross, c='red', s=30, zorder=5)
    ax.set_xlim(-1.5, 1.5)
    ax.set_ylim(-1.5, 1.5)
    ax.set_aspect('equal')
    ax.set_title(f'$h_\\times$, phase={phase:.2f}')

fig.suptitle('Gravitational Wave Polarisations on a Ring of Test Particles',
             fontsize=14, y=1.02)
plt.tight_layout()
plt.show()

In [ ]:
# --- Chirp signal from an inspiraling binary ---
# Simplified chirp waveform

# Chirp mass in solar masses
M_chirp = 30.0  # ~GW150914

# Time to coalescence (in arbitrary units, scaled for display)
t_coal = 1.0  # coalescence at t=0
t = np.linspace(-t_coal, -0.01, 5000)
tau = -t  # time before coalescence

# Frequency evolution: f ~ tau^(-3/8)
f0 = 30.0  # Hz at start
f_gw = f0 * (tau / tau[0])**(-3.0/8)

# Amplitude evolution: A ~ tau^(-1/4)
A0 = 1.0
A_gw = A0 * (tau / tau[0])**(-1.0/4)

# Waveform: h(t) = A(t) * cos(2*pi * integral of f)
# Approximate phase
dt = t[1] - t[0]
phase = np.cumsum(2 * np.pi * f_gw * dt)
h_chirp = A_gw * np.cos(phase)

fig, axes = plt.subplots(3, 1, figsize=(12, 8), sharex=True)

axes[0].plot(t, h_chirp, 'b-', linewidth=0.5)
axes[0].set_ylabel('Strain $h(t)$')
axes[0].set_title('Gravitational Wave Chirp Signal')

axes[1].plot(t, f_gw, 'r-', linewidth=2)
axes[1].set_ylabel('Frequency (Hz)')
axes[1].set_title('Frequency Evolution')

axes[2].plot(t, A_gw, 'g-', linewidth=2)
axes[2].set_ylabel('Amplitude')
axes[2].set_xlabel('Time before coalescence')
axes[2].set_title('Amplitude Evolution')

plt.tight_layout()
plt.show()

print(f'Initial frequency: {f_gw[0]:.1f} Hz')
print(f'Final frequency:   {f_gw[-1]:.1f} Hz')
print(f'Frequency increase: {f_gw[-1]/f_gw[0]:.1f}x')

---
## 11. Cosmology — The FRW Universe

### 11.1 Friedmann–Robertson–Walker metric

$$ds^2 = -c^2 dt^2 + a(t)^2\left[\frac{dr^2}{1-kr^2} + r^2 d\Omega^2\right]$$

where $a(t)$ is the **scale factor** and $k \in \{-1, 0, +1\}$ is the spatial curvature.

### 11.2 Friedmann equations

$$H^2 \equiv \left(\frac{\dot{a}}{a}\right)^2 = \frac{8\pi G}{3}\rho - \frac{kc^2}{a^2} + \frac{\Lambda c^2}{3}$$

$$\frac{\ddot{a}}{a} = -\frac{4\pi G}{3}\left(\rho + \frac{3p}{c^2}\right) + \frac{\Lambda c^2}{3}$$

### 11.3 Density parameters

| Component | $w = p/\rho c^2$ | Scales as |
|---|---|---|
| Radiation | $1/3$ | $a^{-4}$ |
| Matter | $0$ | $a^{-3}$ |
| Dark energy ($\Lambda$) | $-1$ | constant |

> **Physicist's Intuition:** The Friedmann equations are just the Einstein field
> equations applied to a homogeneous, isotropic universe. The scale factor $a(t)$
> tells us how the universe stretches over time. Today's observed acceleration
> requires dark energy ($\Lambda > 0$).

In [ ]:
# === FRIEDMANN EQUATION INTEGRATION ===
# Integrate da/dt = a * H(a) for different cosmologies

# Current values (normalised)
H0 = 1.0  # Hubble constant (normalised)

def friedmann_rhs(t, a, Omega_m, Omega_r, Omega_L):
    """da/dt for flat FRW universe."""
    if a <= 0:
        return 0.0
    H2 = H0**2 * (Omega_r / a**4 + Omega_m / a**3 + Omega_L)
    if H2 < 0:
        return 0.0
    return a * np.sqrt(H2)

cosmologies = {
    'Matter only': (1.0, 0.0, 0.0),
    'Radiation only': (0.0, 1.0, 0.0),
    'Lambda CDM': (0.3, 0.0001, 0.7),
    'De Sitter': (0.0, 0.0, 1.0),
}

fig, axes = plt.subplots(1, 2, figsize=(14, 6))

ax = axes[0]
for name, (Om, Or, OL) in cosmologies.items():
    sol = solve_ivp(friedmann_rhs, [0, 3.0], [0.01],
                    args=(Om, Or, OL), max_step=0.01, rtol=1e-8)
    ax.plot(sol.t, sol.y[0], linewidth=2, label=name)

ax.set_xlabel('Time $t / H_0^{-1}$')
ax.set_ylabel('Scale factor $a(t)$')
ax.set_title('Evolution of the Scale Factor')
ax.legend()
ax.set_ylim(0, 5)

# Analytical checks
ax = axes[1]
t_ana = np.linspace(0.01, 3.0, 200)

# Matter-dominated: a ~ t^(2/3)
a_matter = (1.5 * H0 * t_ana)**(2.0/3)
a_matter /= a_matter[-1]  # normalise

# Radiation-dominated: a ~ t^(1/2)
a_rad = (2.0 * H0 * t_ana)**(0.5)
a_rad /= a_rad[-1]

# De Sitter: a ~ exp(H*t)
a_dS = 0.01 * np.exp(H0 * t_ana)

ax.plot(t_ana, a_matter, 'b-', linewidth=2, label=r'Matter: $a \propto t^{2/3}$')
ax.plot(t_ana, a_rad, 'r-', linewidth=2, label=r'Radiation: $a \propto t^{1/2}$')
ax.plot(t_ana, a_dS, 'g-', linewidth=2, label=r'De Sitter: $a \propto e^{Ht}$')
ax.set_xlabel('Time $t$')
ax.set_ylabel('Scale factor $a(t)$')
ax.set_title('Analytical Solutions')
ax.set_ylim(0, 5)
ax.legend()

plt.tight_layout()
plt.show()

In [ ]:
# --- Hubble parameter and deceleration parameter ---
# Lambda-CDM cosmology
Omega_m = 0.3
Omega_r = 9e-5
Omega_L = 0.7

a_range = np.linspace(0.1, 2.0, 200)
z_range = 1.0 / a_range - 1  # redshift

# H(a) / H0
H_ratio = np.sqrt(Omega_r / a_range**4 + Omega_m / a_range**3 + Omega_L)

# Deceleration parameter: q = -a*a_ddot/a_dot^2
q = 0.5 * Omega_m / a_range**3 + Omega_r / a_range**4 - Omega_L
q /= H_ratio**2

fig, axes = plt.subplots(1, 2, figsize=(14, 5))

axes[0].plot(z_range, H_ratio, 'b-', linewidth=2)
axes[0].set_xlabel('Redshift $z$')
axes[0].set_ylabel('$H(z) / H_0$')
axes[0].set_title('Hubble Parameter')
axes[0].set_xlim(-0.5, 9)
axes[0].invert_xaxis()

axes[1].plot(z_range, q, 'r-', linewidth=2)
axes[1].axhline(0, color='gray', linewidth=0.5, linestyle='--')
axes[1].set_xlabel('Redshift $z$')
axes[1].set_ylabel('$q(z)$')
axes[1].set_title('Deceleration Parameter')
axes[1].set_xlim(-0.5, 9)
axes[1].invert_xaxis()

plt.tight_layout()
plt.show()

# Find transition redshift where q=0
try:
    z_trans = brentq(lambda z: 0.5*Omega_m*(1+z)**3 + Omega_r*(1+z)**4 - Omega_L -
                     (0.5*Omega_m*(1+z)**3 + Omega_r*(1+z)**4 - Omega_L), -0.5, 0.0)
except Exception:
    pass

# Simpler: find where q crosses zero
idx_trans = np.argmin(np.abs(q))
z_trans = z_range[idx_trans]
print(f'Transition redshift (q=0): z ~ {z_trans:.2f}')
print(f'Universe accelerating for z < {z_trans:.2f}')

---
## 12. Summary & Connections

| Topic | Central equation | Section |
|---|---|---|
| Manifolds | Charts, Jacobians | §2 |
| Metric tensor | $ds^2 = g_{\mu\nu}dx^\mu dx^\nu$ | §3 |
| Christoffel symbols | $\Gamma^\lambda_{\mu\nu} = \frac{1}{2}g^{\lambda\sigma}(\partial g + \cdots)$ | §4 |
| Geodesics | $\ddot{x}^\mu + \Gamma^\mu_{\alpha\beta}\dot{x}^\alpha\dot{x}^\beta = 0$ | §5 |
| Curvature | Riemann, Ricci, Gaussian $K$ | §6 |
| Einstein equations | $G_{\mu\nu} + \Lambda g_{\mu\nu} = 8\pi G\,T_{\mu\nu}$ | §7 |
| Schwarzschild | $ds^2 = -(1-r_s/r)dt^2 + \cdots$ | §8 |
| Gravitational lensing | $\Delta\phi = 4GM/(c^2 b)$ | §9 |
| Gravitational waves | $\Box \bar{h}_{\mu\nu} = 0$ | §10 |
| Friedmann cosmology | $H^2 = (8\pi G/3)\rho + \Lambda/3$ | §11 |

### Connections to other notebooks

- **Matrix calculus (03):** Tensor algebra and index manipulation
- **Measure theory (05):** Integration on manifolds
- **Quantum mechanics (06):** Quantum field theory in curved spacetime
- **Statistical foundations (01):** Bayesian inference in cosmological parameter estimation

### Further reading

- Carroll, *Spacetime and Geometry* (2019)
- Misner, Thorne & Wheeler, *Gravitation* (1973)
- Wald, *General Relativity* (1984)
- Weinberg, *Gravitation and Cosmology* (1972)